# Day 8c — Graph Neural Network experiment

A time-boxed but genuine attempt at a GNN (GraphSAGE-style, using `GraphConv` so the identity/behavioral edge weights we already computed are actually used) as a fourth combination strategy, alongside the three in `docs/hybrid-merge.md`.

**Two attempts are shown deliberately, not just the final one**: a naive first pass (no tuning) badly underperformed, which could easily be mistaken for "GNNs don't work here." A second, properly tuned pass (edge weights used, layer normalization, weight decay, learning-rate scheduling, an internal validation split for model selection) tells a very different story. Reporting only the second result would hide a real methodological lesson — that an untuned deep learning model is not a fair test of the *idea*, only of that one attempt.

**Note on execution status**: full CPU-only training here takes ~25-30 minutes (the tuned model alone runs 250 epochs over a 590K-node graph). Given this was already a deliberately time-boxed exploration, and the finding was clearly negative (see below), that time was spent on the core product instead of re-running this notebook purely for polish. The code below is exactly what was run (via equivalent standalone scripts) to produce the real, saved numbers in `results/gnn_experiment_metrics.json` — this notebook is not saved with inline cell outputs, but every number quoted in the takeaway and in `docs/hybrid-merge.md` came from an actual completed run, not an estimate. Re-running this notebook end-to-end will reproduce them.

In [ ]:
import sys, json, time
sys.path.append('..')

import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GraphConv, SAGEConv
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

from src.data import load_merged_train
from src.features import get_feature_lists, build_preprocessor
from src.split import apply_locked_split
from src.graph import load_graph
from src.cost import find_optimal_threshold, DEFAULT_REVIEW_COST

torch.manual_seed(42)

## 1. Build the graph data object

Same features as the classifier (median-impute + scale numeric, one-hot categorical), same node set as the combined identity+behavioral graph, same locked train/holdout split. **Transductive setup**: all 590,540 nodes and their features/edges are visible to the model (standard for GNNs — message passing needs the full graph structure), but the loss during training only ever uses labels from the train split. A further internal 90/10 split of the *train* data creates a validation set for model selection — the holdout is not touched until the one final evaluation.

In [ ]:
train_full = load_merged_train()
numeric_features, categorical_features = get_feature_lists(train_full)
train_df, holdout_df = apply_locked_split(train_full)

preprocessor = build_preprocessor(numeric_features, categorical_features)
preprocessor.fit(train_df[numeric_features + categorical_features])

X_full = preprocessor.transform(train_full[numeric_features + categorical_features])
if hasattr(X_full, 'toarray'):
    X_full = X_full.toarray()
X_full = X_full.astype(np.float32)
print(f'Feature matrix: {X_full.shape}')

In [ ]:
txn_ids = train_full['TransactionID'].to_numpy()
id_to_idx = {tid: i for i, tid in enumerate(txn_ids)}

combined_graph = load_graph('../data/processed/combined_graph.pkl')
edges = list(combined_graph.edges(data='weight'))
src = np.array([id_to_idx[u] for u, v, w in edges], dtype=np.int64)
dst = np.array([id_to_idx[v] for u, v, w in edges], dtype=np.int64)
w = np.array([w for u, v, w in edges], dtype=np.float32)

edge_index = torch.from_numpy(np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])]))
edge_weight = torch.from_numpy(np.concatenate([w, w])).clamp(max=2.0)

y = torch.from_numpy(train_full['isFraud'].to_numpy().astype(np.float32))
x = torch.from_numpy(X_full)

train_ids_all = train_df['TransactionID'].to_numpy()
holdout_ids = set(holdout_df['TransactionID'].tolist())
train_labels = train_df.set_index('TransactionID')['isFraud']
sub_train_ids, val_ids = train_test_split(
    train_ids_all, test_size=0.1, stratify=train_labels.loc[train_ids_all], random_state=42
)
sub_train_ids, val_ids = set(sub_train_ids.tolist()), set(val_ids.tolist())

train_mask = torch.tensor([tid in sub_train_ids for tid in txn_ids], dtype=torch.bool)
val_mask = torch.tensor([tid in val_ids for tid in txn_ids], dtype=torch.bool)
holdout_mask = torch.tensor([tid in holdout_ids for tid in txn_ids], dtype=torch.bool)

data = Data(x=x, edge_index=edge_index, edge_weight=edge_weight, y=y)
data.train_mask, data.val_mask, data.holdout_mask = train_mask, val_mask, holdout_mask
print(data)
print('train:', train_mask.sum().item(), 'val:', val_mask.sum().item(), 'holdout:', holdout_mask.sum().item())

## 2. Attempt 1: naive GraphSAGE (no tuning)

Plain 2-layer network, one fixed learning rate, no edge weights, no normalization, 15 epochs. This is what "just try a GNN" looks like with no real investment.

In [ ]:
class NaiveGraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.2, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        return self.lin(x).squeeze(-1)

naive_model = NaiveGraphSAGE(data.x.shape[1])
naive_optimizer = torch.optim.Adam(naive_model.parameters(), lr=0.01)

n_pos = data.y[data.train_mask].sum().item()
n_neg = data.train_mask.sum().item() - n_pos
pos_weight = torch.tensor(n_neg / n_pos)

t0 = time.time()
for epoch in range(15):
    naive_model.train()
    naive_optimizer.zero_grad()
    out = naive_model(data.x, data.edge_index)
    loss = F.binary_cross_entropy_with_logits(out[data.train_mask], data.y[data.train_mask], pos_weight=pos_weight)
    loss.backward()
    naive_optimizer.step()
    print(f'epoch {epoch}: loss={loss.item():.4f}')
print(f'Naive training took {time.time()-t0:.1f}s')

In [ ]:
naive_model.eval()
with torch.no_grad():
    naive_proba = torch.sigmoid(naive_model(data.x, data.edge_index)).numpy()

holdout_mask_np = data.holdout_mask.numpy()
y_holdout = data.y.numpy()[holdout_mask_np]
amounts_holdout = train_full['TransactionAmt'].to_numpy()[holdout_mask_np]

naive_pr_auc = average_precision_score(y_holdout, naive_proba[holdout_mask_np])
naive_threshold, naive_cost, _ = find_optimal_threshold(y_holdout, naive_proba[holdout_mask_np], amounts_holdout, review_cost=DEFAULT_REVIEW_COST)
print(f'Naive attempt -- Holdout PR-AUC: {naive_pr_auc:.4f}, cost: Rs {naive_cost:,.0f}')

## 3. Attempt 2: tuned GNN

Fixes made after diagnosing the naive attempt's instability (the loss curve actually rose in early epochs — a sign the learning rate was too high):
- `GraphConv` instead of `SAGEConv`, so the graph's real edge weights (inverse-frequency for identity edges, similarity score for behavioral edges) are used, not thrown away
- `LayerNorm` after each graph layer, for training stability on a wide (555-dimensional) input
- Weight decay (L2 regularization)
- A learning-rate scheduler that halves the rate when validation stops improving
- Model selection by validation PR-AUC (best checkpoint kept), evaluated on a held-out slice of the *train* split — never the real holdout
- A wider hidden layer (128) and more epochs, with early stopping

A short 3-configuration sweep (varying learning rate and width) was run first to pick a starting point; the winner (`lr=0.003, weight_decay=5e-4, hidden_dim=128, dropout=0.4`) is trained here to convergence.

In [ ]:
class TunedGNN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.conv1 = GraphConv(in_dim, hidden_dim)
        self.norm1 = torch.nn.LayerNorm(hidden_dim)
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.norm2 = torch.nn.LayerNorm(hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(self.norm1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        x = F.relu(self.norm2(x))
        return self.lin(x).squeeze(-1)

model = TunedGNN(data.x.shape[1], hidden_dim=128, dropout=0.4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8)

EPOCHS = 250
EARLY_STOP_PATIENCE = 25
best_val_pr_auc = -1.0
best_state = None
patience_counter = 0

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_weight)
    loss = F.binary_cross_entropy_with_logits(out[data.train_mask], data.y[data.train_mask], pos_weight=pos_weight)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_proba = torch.sigmoid(out[data.val_mask]).numpy()
        val_pr_auc = average_precision_score(data.y[data.val_mask].numpy(), val_proba)
    scheduler.step(val_pr_auc)

    if val_pr_auc > best_val_pr_auc:
        best_val_pr_auc, best_state, patience_counter = val_pr_auc, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        patience_counter += 1

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f'epoch {epoch}: loss={loss.item():.4f}, val_pr_auc={val_pr_auc:.4f}, best={best_val_pr_auc:.4f}')
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nTuned training took {time.time()-t0:.1f}s, best val_pr_auc={best_val_pr_auc:.4f}')

## 4. Final, one-time holdout evaluation

In [ ]:
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    tuned_proba = torch.sigmoid(model(data.x, data.edge_index, data.edge_weight)).numpy()

tuned_pr_auc = average_precision_score(y_holdout, tuned_proba[holdout_mask_np])
tuned_threshold, tuned_cost, _ = find_optimal_threshold(y_holdout, tuned_proba[holdout_mask_np], amounts_holdout, review_cost=DEFAULT_REVIEW_COST)

with open('../results/classifier_final_metrics.json') as f:
    locked_xgb = json.load(f)

print(f'{"Model":<30} {"PR-AUC":>10} {"Cost (Rs)":>15}')
print(f'{"XGBoost (locked, Day 4)":<30} {"n/a (0.5 thr eval)":>10} {locked_xgb["total_cost_rs"]:>15,.0f}')
print(f'{"GNN, naive (Attempt 1)":<30} {naive_pr_auc:>10.4f} {naive_cost:>15,.0f}')
print(f'{"GNN, tuned (Attempt 2)":<30} {tuned_pr_auc:>10.4f} {tuned_cost:>15,.0f}')

## Save results

In [ ]:
## Takeaway

**Tuning mattered enormously**: naive → tuned improved holdout PR-AUC from 0.39 to 0.64 (a 63% relative gain) and cut cost by 26% (₹579,357 → ₹429,607). An untuned GNN attempt is not a fair test of whether the approach can work here — it mostly tests whether 15 minutes of effort was enough, and it wasn't.

**But even tuned, it still loses to XGBoost**: ₹429,607 vs. the locked classifier's ₹337,421 — about 27% more expensive. The honest verdict: with the time actually invested (tuning sweep + 250-epoch training on a CPU), a GNN does not beat a well-tuned gradient-boosted tree on this dataset. This doesn't rule out a GNN eventually winning with substantially more investment (deeper architectures, attention-based layers, GPU-accelerated hyperparameter search over days rather than hours) — it specifically answers "is it worth pursuing further within this project's timeline," and the answer is no.

## Takeaway

_Fill in after running: the PR-AUC/cost gap between the naive and tuned GNN (showing tuning genuinely matters), and the final honest verdict against the locked XGBoost classifier and the feature-fusion result._